<a href="https://colab.research.google.com/github/kraj9173/CV-Project-BYOP/blob/main/BinaryClassification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import kagglehub

##loading the dataset in colab enviroment
path = kagglehub.dataset_download("marwaelsayedkhalil/image-classification-dataset")

print("Path to dataset files:", path)

100%|██████████| 43.6M/43.6M [00:00<00:00, 141MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/marwaelsayedkhalil/image-classification-dataset/versions/1


In [ ]:
##steps preprocessing
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
##splitting data into trining and validation dataset
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406],
                             [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406],
                             [0.229, 0.224, 0.225])
    ]),
}

train_dir = os.path.join(path, 'training_set')
val_dir = os.path.join(path, 'test_set')

image_datasets = {
    'train': datasets.ImageFolder(train_dir, data_transforms['train']),
    'val': datasets.ImageFolder(val_dir, data_transforms['val'])
}

dataloaders = {
    'train': DataLoader(image_datasets['train'], batch_size=16, shuffle=True, num_workers=2),
    'val': DataLoader(image_datasets['val'], batch_size=16, shuffle=False, num_workers=2)
}
##finding number of class
print(f"Classes found: {image_datasets['train'].classes}")

Using device: cuda
Classes found: ['cats', 'dogs']


In [ ]:
import torch.nn as nn
import torch.optim as optim
from torchvision import models
import copy
import gc
##loading model
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 2)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

num_epochs = 10

best_model_wts = copy.deepcopy(model.state_dict())
best_acc = 0.0

for epoch in range(num_epochs):
    print(f'\nEpoch {epoch+1}/{num_epochs}')
    print('-' * 15)

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    for phase in ['train', 'val']:
        if phase == 'train':
            model.train()
        else:
            model.eval()

        running_loss = 0.0
        running_corrects = 0

        for inputs, labels in dataloaders[phase]:
            inputs = inputs.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            with torch.set_grad_enabled(phase == 'train'):
                outputs = model(inputs)
                _, preds = torch.max(outputs, 1)
                loss = criterion(outputs, labels)

                if phase == 'train':
                    loss.backward()
                    optimizer.step()

            running_loss += loss.item() * inputs.size(0)
            running_corrects += torch.sum(preds == labels.data)

        epoch_loss = running_loss / len(image_datasets[phase])
        epoch_acc = running_corrects.double() / len(image_datasets[phase])

        print(f'{phase.capitalize()} Loss: {epoch_loss:.4f} | Acc: {epoch_acc:.4f}')

        if phase == 'val' and epoch_acc > best_acc:
            best_acc = epoch_acc
            best_model_wts = copy.deepcopy(model.state_dict())
            print(f"best model saved(Accuracy: {best_acc:.4f}) ***")

print(f"\nTraining complete! Best Validation Accuracy: {best_acc:.4f}")

 ##saving best model
model.load_state_dict(best_model_wts)
save_path = 'best_resnet18_cats_dogs.pth'
torch.save(model.state_dict(), save_path)
print(f"Best model successfully saved to '{save_path}'")


Epoch 1/10
---------------
Train Loss: 0.3710 | Acc: 0.8420
Val Loss: 0.2330 | Acc: 0.8995
best model saved(Accuracy: 0.8995) ***

Epoch 2/10
---------------
Train Loss: 0.2375 | Acc: 0.9042
Val Loss: 0.3131 | Acc: 0.8529

Epoch 3/10
---------------
Train Loss: 0.1996 | Acc: 0.9198
Val Loss: 0.2223 | Acc: 0.9167
best model saved(Accuracy: 0.9167) ***

Epoch 4/10
---------------
Train Loss: 0.2274 | Acc: 0.9042
Val Loss: 0.2383 | Acc: 0.9020

Epoch 5/10
---------------
Train Loss: 0.1444 | Acc: 0.9434
Val Loss: 0.1860 | Acc: 0.9338
best model saved(Accuracy: 0.9338) ***

Epoch 6/10
---------------
Train Loss: 0.1520 | Acc: 0.9434
Val Loss: 0.1598 | Acc: 0.9510
best model saved(Accuracy: 0.9510) ***

Epoch 7/10
---------------
Train Loss: 0.1172 | Acc: 0.9552
Val Loss: 0.3594 | Acc: 0.8652

Epoch 8/10
---------------
Train Loss: 0.1267 | Acc: 0.9565
Val Loss: 0.2056 | Acc: 0.9338

Epoch 9/10
---------------
Train Loss: 0.1197 | Acc: 0.9527
Val Loss: 0.3055 | Acc: 0.8922

Epoch 10/10
---